# 16 · Terrain and water-level exploration in 3D

**Question:** How much of a dramatic terrain scene is data, and how much is display exaggeration?

**Runtime:** Python / Pyodide in JupyterLite. Choose **Run → Run All Cells**.
Each notebook is independent; start with a fresh kernel. Initial runtime/package downloads need internet.
The initial baseline uses **synthetic educational fixtures** near a Gaborone-area anchor.
Extended investigations explicitly identify bundled real OSM, Sentinel or terrain extracts where used;
these do not validate the synthetic population, mobility, exposure or scenario assumptions.

**Astra experiment:** Have Astra compare the 3D rendering with a cell-based area calculation and ask Codex for controls that preserve vertical units.
Run the numerical baseline first, then give Astra the exported evidence and the prompt below.
The baseline does not call or run GPT-6 Astra. No API key is needed.


In [ ]:
import sys, json, math, time
from pathlib import Path
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "matplotlib"])
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from twin import *
style()
print("Runtime:", sys.platform, "| data: synthetic | seed: 42")


## Surface, datum and screening plane · 30 minutes
**Learn:** construct a 3D surface from a raster, label a vertical datum, and compare a threshold area with a visual impression.
Plotly supplies an orbitable scientific surface rather than a tiled geographic basemap. Coordinates are local
east/north metres; elevations use an invented datum. Each of 40 × 40 cells represents 50 × 50 m, sampled at
its center. The surface interpolates between centers for display; area uses full cell counts.
**Predict:** does changing vertical exaggeration change the area below the water plane?
This threshold model omits drainage and hydraulic connectivity. It is not a flood forecast.


In [ ]:
from scenes import export_scene, scene_frame
pixel_m=50.; axis=(np.arange(40)+.5)*pixel_m
xx,yy=np.meshgrid(axis,axis)
elevation=1000+.003*xx+.002*yy-4*np.exp(-((xx-1100)/250)**2)
water_level=1001.
below=elevation<water_level
area_m2=int(below.sum())*pixel_m**2
levels=np.arange(998,1008.5,.5)
areas=np.array([np.count_nonzero(elevation<level)*pixel_m**2 for level in levels])
assert (np.diff(areas)>=0).all() and 0<=area_m2<=2000**2
assert elevation.shape==(40,40) and len(axis)*pixel_m==2000
print('Cells below plane:',int(below.sum()),'| screening area:',area_m2,'m²')
payload=dict(x=axis.tolist(),y=axis.tolist(),z=elevation.tolist(),pixel_m=pixel_m,
             synthetic=True,datum='invented local vertical reference',units='metres')
export_json('16_terrain.json',payload)


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,5))
image=axes[0].imshow(elevation,origin='lower',extent=[0,2000,0,2000],cmap='terrain',vmin=995,vmax=1012)
axes[0].contour(xx,yy,elevation,levels=[water_level],colors=['#176caf'])
map_axes(axes[0],'Plan view and 1001 m contour'); fig.colorbar(image,ax=axes[0],label='Synthetic elevation (m)')
axes[1].plot(levels,areas/1e6,marker='o',color='#087f8c')
axes[1].set(xlabel='Water screening level (m)',ylabel='Area below plane (km²)',title='Cell-based sensitivity; not hydraulic extent')
plt.tight_layout(); plt.show()


## Use and share the interactive scene
The scene appears below in a trusted notebook output. If your notebook viewer blocks the iframe,
download the HTML from `exports/` and open it in a browser. All data are embedded, but the pinned
JavaScript/CSS renderer still downloads from a CDN; this is not fully offline. WebGL must be available.
The static plot and numerical outputs remain usable when the renderer cannot load.

Camera rotation and vertical exaggeration change the view, not the underlying data. Keep a plan view
beside a perspective view to detect occlusion. Download the notebook, scene and evidence JSON before
clearing browser storage. Do not label synthetic geometry as observed Gaborone buildings.


In [ ]:
document=export_scene('16_terrain_scene.html','terrain',payload)
display(scene_frame(document,'Synthetic terrain and water screening plane'))


## Astra map audit and a Codex surface task
**Astra:** send `16_terrain.json` and screenshots at 1× and 30× vertical exaggeration.
Ask: “How many cell centers lie below 1001 m, and what full-cell area does that represent? Why do the two
views imply different apparent slopes but the same area? Which evidence is missing for a real flood map?”
Grade count and area exactly against the cell calculation; review the scientific explanation separately.

**Codex:** “Add an elevation cross-section at a selected row to `content/scene_templates/terrain.html`.
Keep its y-axis in real metres regardless of 3D exaggeration. Mark the selected row on the scene and test
the cross-section against the corresponding exported array.”

**Acceptance:** water-level increases cannot reduce threshold area; exaggeration cannot change elevations
or cell counts; row zero is south; datum and units remain visible. Record the exact prompt and response before
using the reference calculation to evaluate it.

Reference: [Plotly JavaScript surface plots](https://plotly.com/javascript/3d-surface-plots/).
For georeferenced terrain tiles or a globe, MapLibre terrain or CesiumJS is a further extension requiring
documented elevation sources, licensing and vertical references.


## Extended investigation: Inspect a real terrain chip and an elevation cross-section

Load a bundled Mapzen terrain extract near Gaborone, decoded from Terrarium RGB and downsampled on the desktop. It retains actual sample coordinates, processing notes and source attribution. Work in elevation relative to the chip minimum because this composite’s vertical datum has not been independently verified. The 3D inspector adds a selectable cross-section whose elevations remain in real metres when the view is exaggerated. A below-plane area remains a screening statistic, never a hydraulic flood extent.

**Read the concept map first:** identify the input, method, evidence and limit. Predict the spatial pattern before running the comparison.


In [ ]:
from extension_tools import concept_map, export_investigation
concept_map('Inspect a real terrain chip and an elevation cross-section', ['Terrarium DEM + sample centers', 'Decode / orient / inspect nodata', 'Hillshade + profile + 3D', 'Vertical datum not independently verified'])
plt.show()


In [ ]:
from extension_tools import load_data
from scenes import export_scene, scene_frame
ex_dem=load_data('gaborone_terrain.json'); ex_z=np.array(ex_dem['z']); ex_x=np.array(ex_dem['x']); ex_y=np.array(ex_dem['y'])
assert ex_z.shape==(len(ex_y),len(ex_x)) and np.isfinite(ex_z).all() and (np.diff(ex_x)>0).all() and (np.diff(ex_y)>0).all()
ex_relative=ex_z-ex_z.min(); ex_row=16; ex_level=10.
gy,gx=np.gradient(ex_relative,ex_y,ex_x); ex_hill=np.clip((1-gx-gy)/np.sqrt(3*(1+gx*gx+gy*gy)),0,1)
ex_area=float(np.count_nonzero(ex_relative<ex_level)*ex_dem['pixel_area_m2'])
fig,axes=plt.subplots(1,3,figsize=(14,4)); extent=[ex_x.min(),ex_x.max(),ex_y.min(),ex_y.max()]
axes[0].imshow(ex_hill,origin='lower',extent=extent,cmap='gray'); axes[0].axhline(ex_y[ex_row],color='#c44e52'); axes[0].set(title='Real terrain hillshade + profile row',xlabel='Local east (m)',ylabel='Local north (m)')
image=axes[1].imshow(ex_relative,origin='lower',extent=extent,cmap='terrain'); fig.colorbar(image,ax=axes[1],label='Metres above chip minimum'); axes[1].set(title='Relative elevation, datum caveat retained',xlabel='Local east (m)',ylabel='Local north (m)')
axes[2].plot(ex_x,ex_relative[ex_row],c='#087f8c'); axes[2].axhline(ex_level,c='#318dcc',ls='--'); axes[2].set(title='Cross-section: real vertical units',xlabel='Local east (m)',ylabel='Metres above chip minimum')
fig.text(.01,.01,'Mapzen terrain tiles · USGS SRTM/GMTED2010 / NOAA ETOPO1 · see data provenance for processing and attribution',fontsize=8)
fig.tight_layout(rect=[0,.05,1,1]); extension_figure=fig
ex_payload=dict(x=ex_x.tolist(),y=ex_y.tolist(),z=ex_relative.tolist(),pixel_m=float(np.sqrt(ex_dem['pixel_area_m2'])),relative=True,attribution=ex_dem['provenance']['attribution'],title='Real terrain: relative elevation',description='Mapzen chip near Gaborone; elevations relative to the chip minimum; vertical datum not independently verified.')
ex_html=export_scene('16_real_terrain.html','terrain',ex_payload); display(scene_frame(ex_html,'Real terrain chip and selectable cross-section'))
extension_evidence=dict(provenance=ex_dem['provenance'],profile_m=ex_relative[ex_row].tolist(),cell_count_below_plane=int(np.count_nonzero(ex_relative<ex_level)),cell_area_m2=ex_dem['pixel_area_m2'],plane_m_above_minimum=ex_level)
extension_expected=dict(profile_range_m=float(np.ptp(ex_relative[ex_row])),below_plane_area_m2=ex_area)
extension_task='Calculate profile_range_m and below_plane_area_m2. Use the provided cell-area approximation and do not interpret the screening plane as a flood forecast.'


In [ ]:
extension_prompt,extension_reference=export_investigation('16',extension_evidence,extension_expected,extension_task,extension_figure)
plt.show()
print('Send only 16_extension_prompt.json and optionally 16_extension_map.png to Astra. Keep the reference in this kernel.')


### Ask, check and explain the spatial result
Download `16_extension_prompt.json` and `16_extension_map.png`. Send the evidence in a fresh Astra
session with the exact task and output fields. For a separate image-only experiment, supply only the image
and task wording, record that condition, and allow clarification if the image does not contain enough information.
Never send the reference or this complete notebook for a blind trial.

Paste the unedited response below or upload a UTF-8 response file. The default is deliberately empty:
**no model trial has run**. After recording a real answer, set `extension_source='MODEL'` and complete the metadata.
The numeric tolerance is 0.01 in each requested field's declared units; labels and booleans require exact agreement.
Keep that rule fixed before examining a response. Inspect the explanation separately for unsupported claims.

**Visual reflection:** point to one map pattern, cite the numerical check that supports it, and name one thing
the visual cannot establish. Re-run one parameter change and preserve both maps. The implementation above
is available for inspection and modification; it is not a measured claim about model accuracy.


In [ ]:
from experiments import grade_answer
extension_source='NOT RUN'
extension_model=''; extension_date=''; extension_interface=''; extension_condition='text-and-map'
extension_response_file=''; extension_response_text=''
if extension_response_file:
    extension_response_text=Path(extension_response_file).read_text(encoding='utf-8')
if extension_source=='MODEL':
    assert extension_model and extension_date and extension_interface and extension_response_text, 'Record the actual model, date, interface and response'
extension_tolerances={key:.01 for key,value in extension_reference.items() if type(value) in (int,float)}
extension_grade=grade_answer(extension_response_text,extension_reference,extension_tolerances) if extension_response_text else dict(status='NOT RUN')
display(extension_grade)
export_json('16_extension_trial.json',dict(source=extension_source,model=extension_model,date=extension_date,
    interface=extension_interface,condition=extension_condition,prompt=extension_prompt,response_text=extension_response_text,
    tolerances=extension_tolerances,reference=extension_reference,grade=extension_grade))


## Save the investigation
Download the edited notebook, evidence, map and trial record from `exports/`. Browser storage does not
commit changes to GitHub. Keep data attribution and the distinction between observed data, synthetic
assumptions and actual model responses when sharing. The extended code and its checks above are part of
this notebook; use them as the starting point for your next controlled comparison.
